[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/03_baseline_rf.ipynb)

# 03 — Baseline Random Forest (HOG Features)
Establecemos el baseline clásico con Random Forest sobre características HOG para comparar con las arquitecturas de Deep Learning.

In [ ]:
import os, sys, h5py
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
from skimage.feature import hog
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score

# ── Configuración Global ──────────────────────────────────────────────────
N_SAMPLES = 1000  # Cantidad de parches para el baseline (ajustar según RAM)
DATA_ROOT = Path('/content/landslide4sense') # Ajustar según Notebook 01/02

## 1. Extracción de características HOG
Transformamos los datos espectrales en descriptores de gradiente para capturar texturas de deslizamientos.

In [ ]:
img_dir  = DATA_ROOT / "TrainData" / "img"
mask_dir = DATA_ROOT / "TrainData" / "mask"
files    = sorted(list(img_dir.glob("*.h5")))[:N_SAMPLES]

X, y = [], []
for f in tqdm(files, desc="Extrayendo HOG"):
    mf = mask_dir / f.name
    if not mf.exists(): continue
    
    with h5py.File(f,  "r") as hf: patch = hf[list(hf.keys())[0]][()]
    with h5py.File(mf, "r") as hf: mask  = hf[list(hf.keys())[0]][()]
    
    # Usamos bandas RGB [B4, B3, B2] para HOG estándar
    rgb = patch[:,:,[5,4,3]] 
    rgb = ((rgb - rgb.min()) / (rgb.ptp() + 1e-8) * 255).astype("uint8")
    
    # HOG Descriptor
    feats = hog(rgb, pixels_per_cell=(16,16), cells_per_block=(2,2), 
                channel_axis=-1, feature_vector=True)
    
    X.append(feats)
    y.append(int(mask.max() > 0)) # Clasificación binaria: ¿Hay deslizamiento?

X, y = np.array(X), np.array(y)
print(f"\nFeatures HOG extraídos: {X.shape}")
print(f"Positivos: {y.sum()} ({100*y.mean():.1f}%) | Negativos: {len(y)-y.sum()}")

## 2. Entrenamiento y Validación Cruzada (5-Fold)

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
metrics = {'f1': [], 'auc': [], 'prec': [], 'rec': []}

for fold, (tr, va) in enumerate(skf.split(X, y)):
    rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)
    rf.fit(X[tr], y[tr])
    
    probs = rf.predict_proba(X[va])[:,1]
    preds = (probs >= 0.5).astype(int)
    
    metrics['f1'].append(f1_score(y[va], preds, zero_division=0))
    metrics['auc'].append(roc_auc_score(y[va], probs))
    metrics['prec'].append(precision_score(y[va], preds, zero_division=0))
    metrics['rec'].append(recall_score(y[va], preds, zero_division=0))
    
    print(f"Fold {fold}: F1={metrics['f1'][-1]:.4f}")

print(f"\nResumen CV: F1={np.mean(metrics['f1']):.4f} ± {np.std(metrics['f1']):.4f}")

## 3. Importancia de Características HOG

In [ ]:
rf.fit(X, y)
importances = rf.feature_importances_
top_idx = np.argsort(importances)[-20:]

plt.figure(figsize=(10, 5))
plt.barh(range(20), importances[top_idx], color='skyblue')
plt.yticks(range(20), [f"Feature {i}" for i in top_idx])
plt.title("Top 20 Importancia de Características (HOG)")
plt.xlabel("Importancia Gini")
plt.show()